# TP0 — Premiers pas (corrigé)
**CNFSDP — Introduction au Machine Learning** · L3 S5 · 2026-2027 · Rois Céti DIMBAMBA

Ce notebook accompagne le **chapitre 0** (polycopié). Il vérifie votre installation, reproduit les programmes de la section 6, puis propose quatre exercices.

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rdimbamba/ml-cnfsdp/blob/main/notebooks/corriges/TP0_corrige.ipynb)

## 0. Vérifier l'installation

In [1]:
import sys
import numpy as np, pandas as pd, sklearn
print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

Python 3.13.16 | NumPy 2.5.3 | pandas 3.0.5 | scikit-learn 1.9.1


## 1. Programmes de la section 6 du polycopié

### Matrice de covariance et valeurs propres

In [2]:
import numpy as np

# Matrice de covariance et valeurs propres (sections 1)
X = np.array([[1, 2], [2, 2], [3, 5]])        # 3 individus, 2 variables
Xc = X - X.mean(axis=0)                        # tableau centré
S = Xc.T @ Xc / len(X)                         # covariance empirique
print(S.round(3))                              # [[0.667 1.] [1. 2.]]
lam, U = np.linalg.eigh(S)                     # valeurs propres croissantes
print(lam.round(4))                            # [0.1315 2.5352]
print(U[:, -1].round(3))                       # premier axe de l'ACP

[[0.667 1.   ]
 [1.    2.   ]]
[0.1315 2.5352]
[0.472 0.882]


### Droite des moindres carrés et descente de gradient

In [3]:
import numpy as np

# Droite des moindres carrés (section 2)
x = np.array([1, 2, 3]); y = np.array([2, 2, 5])
b = np.mean((x - x.mean()) * (y - y.mean())) / np.var(x)   # s_xy / s_x²
a = y.mean() - b * x.mean()
print(a, b)                                    # 0.0 1.5
print(np.polyfit(x, y, deg=1).round(4))        # [1.5 0.] : même droite

# Descente de gradient sur L(θ) = (θ - 3)²
theta, eta = 0.0, 0.25
for k in range(6):
    theta = theta - eta * 2 * (theta - 3)
    print(round(theta, 4), end=" ")            # 1.5 2.25 2.625 2.8125 ...

0.0 1.5
[1.5 0. ]
1.5 2.25 2.625 2.8125 2.9062 2.9531 

### Loi des grands nombres et théorème central limite

In [4]:
import numpy as np
rng = np.random.default_rng(2)

# Loi des grands nombres et TCL (section 3)
p, n, R = 0.3, 400, 20_000                     # R sondages de n personnes
props = rng.binomial(n, p, size=R) / n         # R proportions observées
print(props.mean().round(4), props.var().round(6), p * (1 - p) / n)
ecart = 1.96 * np.sqrt(p * (1 - p) / n)
print(np.mean(np.abs(props - p) <= ecart).round(3))   # proche de 0.95

0.3 0.00053 0.000525
0.943


### Moyennes conditionnelles

In [5]:
import numpy as np, pandas as pd
rng = np.random.default_rng(3)

# Moyennes conditionnelles par groupe (section 4)
n = 10_000
milieu = rng.choice(["urbain", "rural"], size=n, p=[0.4, 0.6])
enfants = np.where(milieu == "urbain", rng.poisson(2.5, n), rng.poisson(4.5, n))
df = pd.DataFrame({"milieu": milieu, "enfants": enfants})

print(df.groupby("milieu")["enfants"].mean().round(2))   # ≈ 4.5 et 2.5
part = df["milieu"].value_counts(normalize=True)
moy = df.groupby("milieu")["enfants"].mean()
print(round((part * moy).sum(), 4), round(df["enfants"].mean(), 4))  # espérance totale

milieu
rural     4.50
urbain    2.47
Name: enfants, dtype: float64
3.6835 3.6835


## 2. Exercices
### Exercice 1 — Matrices
Calculer $A^\top A$ et son inverse pour $A=\begin{pmatrix}1&0\\1&1\\1&2\end{pmatrix}$, et comparer avec votre calcul à la main (exercice 1 du polycopié).

In [6]:
A = np.array([[1, 0], [1, 1], [1, 2]])
AtA = A.T @ A
print(AtA)                       # [[3 3] [3 5]]
print(np.linalg.inv(AtA) * 6)    # 6 × inverse = [[5 -3] [-3 3]]

[[3 3]
 [3 5]]
[[ 5. -3.]
 [-3.  3.]]


### Exercice 2 — Formule de Bayes par simulation
Prévalence 1 %, test positif chez 95 % des malades et négatif chez 90 % des non-malades. Simuler un million de personnes et estimer $\mathbb P(M\mid +)$ ; comparer à $0{,}0095/0{,}1085\approx0{,}088$.

In [7]:
rng = np.random.default_rng(11)
N = 1_000_000
malade = rng.random(N) < 0.01
positif = np.where(malade, rng.random(N) < 0.95, rng.random(N) < 0.10)
print(malade[positif].mean(), 0.0095 / 0.1085)

0.0881089749497705 0.08755760368663594


### Exercice 3 — Choix du pas
Pour $L(\theta)=2\theta^2$ (dérivée $4\theta$), faire 50 pas de descente de gradient depuis $\theta_0=1$ avec $\eta=0{,}4$, puis $\eta=0{,}6$. Comparer avec l'exercice 6 du polycopié.

In [8]:
for eta in [0.4, 0.6]:
    theta = 1.0
    for _ in range(50):
        theta = theta - eta * 4 * theta
    print(f"eta = {eta} : theta_50 = {theta:.3e}")   # converge si eta < 0,5

eta = 0.4 : theta_50 = 8.083e-12
eta = 0.6 : theta_50 = 2.025e+07


### Exercice 4 — Un biais qui paie
Pour $\mu=1$, $\sigma=3$, $n=10$, estimer par simulation l'erreur quadratique moyenne de $c\,\bar X_n$ pour $c\in[0{,}5;1{,}2]$, puis comparer le minimum à $c^*=\mu^2/(\mu^2+\sigma^2/n)$.

In [9]:
rng = np.random.default_rng(12)
mu, sigma, n, R = 1.0, 3.0, 10, 200_000
xbar = rng.normal(mu, sigma, size=(R, n)).mean(axis=1)
cs = np.linspace(0.5, 1.2, 15)
eqm = np.array([np.mean((c * xbar - mu) ** 2) for c in cs])
print("c optimal simulé :", cs[eqm.argmin()].round(3), "| c* théorique :", round(mu**2 / (mu**2 + sigma**2 / n), 3))
print("EQM(c*) ≈", eqm.min().round(3), "contre EQM(1) ≈", np.mean((xbar - mu) ** 2).round(3))

c optimal simulé : 0.55 | c* théorique : 0.526
EQM(c*) ≈ 0.476 contre EQM(1) ≈ 0.902
